# 02 p3 development diagnostics

**Permitted partitions: TRAIN ONLY**

Inspect training-only diagnostics without revising the frozen protocol. Real runs fetch the verified archive temporarily; only development rows are exposed.

Protocol: frozen 2026-10-08; provenance uses exact content hashes. Training targets end 2015-02-28; validation ends 2016-02-29. Final test outcomes are not analyzed here.

Prerequisites: open this notebook from the same P4 Git revision you will enter below. In Colab Secrets, set `AIRSENSE_GITHUB_TOKEN` with write access to this repository. Outputs go to `results/<run-id>` on a dedicated Git branch after every checkpoint. Expect setup/download to take minutes; no forecasting model training occurs. If dependency installation requires a runtime restart, restart and rerun setup.


## Runtime bootstrap
Clone a pinned revision in a fresh runtime; preserve existing work on rerun.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/Engr-Daniel/airsense-r.git"
# For Colab: paste the full P4 commit SHA from GitHub when prompted.
# Resume using the SAME revision and run ID as the original execution.
TEST_MODE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_MODE") == "1"
if TEST_MODE:
    ROOT = Path(os.environ["AIRSENSE_NOTEBOOK_TEST_ROOT"]).resolve()
else:
    CODE_REVISION = os.environ.get("AIRSENSE_CODE_REVISION") or input("Full P4 code commit SHA: ").strip()
    if len(CODE_REVISION) != 40 or any(c not in "0123456789abcdef" for c in CODE_REVISION.lower()):
        raise ValueError("Use a full Git commit SHA, not a moving branch name")
    ROOT = Path.cwd() / "airsense-r"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REVISION], check=True)
    else:
        remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
        head = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        if remote != REPOSITORY or head != CODE_REVISION or dirty:
            raise RuntimeError("Existing checkout differs or has edits; preserve it and use a fresh runtime")
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11+ required")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-p4.txt"), "-e", str(ROOT)], check=True)
    print("Code revision:", CODE_REVISION)
print("Workspace:", ROOT)


## Artifact authentication and run identity
The secret is never printed or written into Git configuration. Use the same run ID only for compatible resumes.

In [ ]:
import logging
from airsense_r.artifacts import provenance, json_bytes
from airsense_r.artifacts import source_hash
from airsense_r.checkpoints import open_run
from airsense_r.colab import configure_github_secret

logging.basicConfig(level=logging.INFO)
if source_hash(ROOT) != "5acfc055105c9fa70a98bd938fa76b4df18e4d618fd7e5fcfa9e0c28fb865ee4":
    raise RuntimeError("Notebook and scientific source revisions differ; open the matching notebook revision")
if TEST_MODE:
    RUN_ID = os.environ["AIRSENSE_NOTEBOOK_TEST_RUN"]
    REMOTE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_REMOTE")
else:
    from google.colab import userdata
    configure_github_secret(ROOT, userdata.get("AIRSENSE_GITHUB_TOKEN"))
    RUN_ID = input("Run ID (reuse only to resume compatible checkpoints): ").strip()
    REMOTE = REPOSITORY


## Evidence and validation
Provenance is checked before restoring outputs. Each completed artifact is saved and pushed before continuing.

In [ ]:
from airsense_r.p4 import run_pipeline, synthetic_stream
if TEST_MODE:
    store = open_run(ROOT, RUN_ID, provenance(ROOT, {"notebook": "02", "synthetic_only": True}), REMOTE)
    data = synthetic_stream()
    summary = data.groupby("station")["PM2.5"].agg(["count", "mean"])
    store.save("synthetic-diagnostics.csv", summary.to_csv().encode())
    display(summary)
else:
    output = run_pipeline(ROOT, RUN_ID, remote=REMOTE, training_diagnostics=True)
    print("Verified training summaries saved incrementally:", output)
    from airsense_r.p4 import inspect_training_diagnostics
    display(inspect_training_diagnostics(ROOT, output))


## Interpretation and limitations
Review the displayed evidence before drawing conclusions. Synthetic checks establish implementation behavior, not predictive performance. Training diagnostics cannot justify silently changing frozen settings. Audit missingness is structural evidence. A failed push stops execution with a local checkpoint retained; restart with the same revision/run ID to restore remotely confirmed work. In-flight work not yet uploaded can be lost on runtime termination. See `docs/P4_WORKFLOW.md` for commands and validation limits.